In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.ml.recommendation import ALS
from pyspark.sql.functions import col
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import explode
import pandas as pd
import numpy as np
from pyspark.ml.evaluation import RegressionEvaluator
spark = SparkSession.builder.appName("ALS_recommender").getOrCreate()

### PySark ALS for 100k movie dataset:

In [2]:
#Loading data from CSV files:
url = 'https://raw.githubusercontent.com/stormwhale/data-mines/refs/heads/main/ratings.csv'
url2 = 'https://raw.githubusercontent.com/stormwhale/data-mines/refs/heads/main/movies.csv'
df = pd.read_csv(url)
df_name = pd.read_csv(url2)

df_com = df.merge(df_name, on='movieId', how = 'left')

#Read the CSV in pandas and convert it to Sparkdf:
df_com_sy = spark.createDataFrame(df_com)
df_com_sy.show(5)


+------+-------+------+---------+--------------------+--------------------+
|userId|movieId|rating|timestamp|               title|              genres|
+------+-------+------+---------+--------------------+--------------------+
|     1|      1|   4.0|964982703|    Toy Story (1995)|Adventure|Animati...|
|     1|      3|   4.0|964981247|Grumpier Old Men ...|      Comedy|Romance|
|     1|      6|   4.0|964982224|         Heat (1995)|Action|Crime|Thri...|
|     1|     47|   5.0|964983815|Seven (a.k.a. Se7...|    Mystery|Thriller|
|     1|     50|   5.0|964982931|Usual Suspects, T...|Crime|Mystery|Thr...|
+------+-------+------+---------+--------------------+--------------------+
only showing top 5 rows



In [3]:
#Before train split the data, we will select only relevant columns:
df = df_com_sy.select('userId', 'rating', 'movieId')

#Change score column to integers:
df = df.withColumn('rating', col('rating').cast('float'))

print(df.dtypes)

[('userId', 'bigint'), ('rating', 'float'), ('movieId', 'bigint')]


In [4]:
df.show(5)

+------+------+-------+
|userId|rating|movieId|
+------+------+-------+
|     1|   4.0|      1|
|     1|   4.0|      3|
|     1|   4.0|      6|
|     1|   5.0|     47|
|     1|   5.0|     50|
+------+------+-------+
only showing top 5 rows



Train Split the data for the ALS model:

In [5]:
train, test = df.randomSplit([0.8, 0.2], seed=123)

train_count = train.count()
test_count = test.count()
print(f'Total number of data in training: {train_count}')
print(f'Total number of data in test: {test_count}')


Total number of data in training: 80753
Total number of data in test: 20083


ALS model building:

In [6]:
#Define the ALS model:
als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative = True, #This hyperparameter controls whether negative
    implicitPrefs = False, #This hyperparameter controls whether the feedback is implicit (e.g. modeling for clicks and likes)
    coldStartStrategy = 'drop',
    seed = 123,
    maxIter= 10
    )

#parameters for tuning:
'''
ALS is a matrix factorization algorithm that can be used for collaborative filtering and works well when pairing up with Spark, which handles data using in-memory retrieval.
ALS breaks the user-item matrices (R) into two smaller matrices, U (user factors) and V (item factors).
Then the ALS algorithm alternates between fixing V to solve for U and U to solve for V until the product UVt predicts the known ratings in R as closely as possible.

maxIter controls how many iterations that the ALS will alternate when finding the least error.
rank is the number of latent factors (hidden features) that the model will learn.
regParam is the regularization parameter to help prevent overfitting.
'''
param_grid = ParamGridBuilder().addGrid(als.rank, [10, 20]).addGrid(als.regParam, [0.1, 0.05]).build()

#Set up model evaluation (RMSE):
evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)

#Set up cross-validation:
crossval = CrossValidator(
    estimator = als,
    estimatorParamMaps = param_grid,
    evaluator=evaluator,
    numFolds=3
)

#Fitting the model:
cv_model = crossval.fit(train)
best_als_model = cv_model.bestModel

Evaulate model performance and parameters:

In [7]:
#Print best model parameters:
print('Best model parameters are:')
print(f'Rank: {best_als_model.rank}')
print(f"regParam : {best_als_model._java_obj.parent().getRegParam()}")
print(f"Best model's RMSE: {evaluator.evaluate(cv_model.transform(test))}")

Best model parameters are:
Rank: 20
regParam : 0.1
Best model's RMSE: 0.8824879052860946


In [8]:
user = train.select('userId').distinct()
items = train.select('movieId').distinct()
user_item = user.crossJoin(items) #This cross join creates every possible user and movie combination, preparing for the model predictions.
dfs_pred = best_als_model.transform(user_item)

#Excluding the duplicate user-movie pairs from the training set:
existing = train.select('userId', 'movieId').distinct()
dfs_exc_train = dfs_pred.join(existing, ['userId', 'movieId'], how='left_anti') #Use anti-join to exclude the user-movie pairs


Making recommendations for top 3 movie recommendations for all users:

In [9]:
from os import truncate
user_recs = best_als_model.recommendForAllUsers(3)

exploded_recs = user_recs.select(
    'userId', explode('recommendations').alias('rec')
).select(
    'userId',
    col('rec.movieId').alias('movieId'),
    col('rec.rating').alias('rating')
)

#Select the movie dataframe:
movie_df = df_com_sy.select('movieId', 'title').distinct()

#Join with movie titles:
top3_list = exploded_recs.join(movie_df, 'movieId', 'left')

#Show top-3 movies for all users:
top3_list.orderBy(['userId', 'rating'], ascending=[True, False]).show(truncate=False)



+-------+------+---------+--------------------------------------------------------------------+
|movieId|userId|rating   |title                                                               |
+-------+------+---------+--------------------------------------------------------------------+
|177593 |1     |5.519942 |Three Billboards Outside Ebbing, Missouri (2017)                    |
|171495 |1     |5.4773884|Cosmos                                                              |
|3508   |1     |5.4279504|Outlaw Josey Wales, The (1976)                                      |
|131724 |2     |4.875957 |The Jinx: The Life and Deaths of Robert Durst (2015)                |
|89904  |2     |4.838698 |The Artist (2011)                                                   |
|417    |2     |4.6211257|Barcelona (1994)                                                    |
|5746   |3     |4.9093084|Galaxy of Terror (Quest) (1981)                                     |
|70946  |3     |4.8970046|Troll 2 (1990)

**Reasons to use ALS or ALS with Spark:**

1) Able to reveal and analyze explicit and implicit feedbacks such as clicks, likes, watch time.

2) Highly scalable. PySpark can process millions of user-item data, making it ideal for industrial-scale.

3) Native built-in to work on spare matrices without the need to fill-in missing data.

4) Parallelizable to increase computation efficiency with multi-core CPUs.

**ALS limitations:**

1) ALS struggles with cold-start problem. The user-item matrix needs to have certain amount of data in order to function properly.

2) The hyperparameters and grid search can be computationally expensive. Computers with more CPU cores may benefit from parallelization. As demonstrated in this exercise where the notebook is done in Google Cloud, the computation time is very long and the number of maxItera cannot be too big or the notebook will timeout the session.